# DX-COM Tutorial 1: Beginner

This notebook introduces the complete ONNX-to-DXNN workflow with small, reproducible examples.

You will:

1. verify the DX-COM installation,
2. export and validate a MobileNetV2 ONNX model,
3. create a calibration configuration,
4. compile and inspect a DXNN model, and
5. download an ONNX model and its JSON configuration from the DEEPX Model Zoo and compile it.

This notebook does not modify the SDK source tree. All generated files are stored under `<dx-tutorials>/T05-DX-Compiler`.


## Course map

| Notebook | Main focus |
|---|---|
| Beginner | Installation, ONNX validation, JSON basics, first compile, Model Zoo |
| Intermediate | Calibration quality, hardware PPU, Python API, YOLO26 TopK optimization |
| Advanced | Q-PRO, diagnosis, QXNN resume, QAT, advanced compiler controls |

Complete the notebooks in order unless you already understand DX-COM configuration and calibration.


## 1. Requirements and workspace

DX-COM 2.4.0 supports x86-64 Linux. Use a static ONNX input shape with batch size 1. A practical host has at least 16 GB of RAM and 8 GB of free storage.

The following cell reads the SDK location from `config.json` through `tutorial_paths.py`. It then derives the compiler and virtual-environment paths automatically.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import sys

root_path = os.environ.get("ROOT_PATH")
if not root_path:
    raise EnvironmentError("ROOT_PATH is not set. Start JupyterLab with ./run-jupyter-lab.sh")
%run "$root_path/tutorial_paths.py"
print_tutorial_paths()

DX_COM_DIR = DX_COMPILER_DIR / "dx_com"
DX_COMPILER_VENV = DX_COMPILER_DIR / "venv-dx-compiler-local"
DXCOM_PATH = DX_COMPILER_VENV / "bin" / "dxcom"
DXCOM_PYTHON = DX_COMPILER_VENV / "bin" / "python"

for required_path in (DX_COM_DIR, DXCOM_PATH, DXCOM_PYTHON):
    if not required_path.exists():
        raise FileNotFoundError(f"Required DX-COM path does not exist: {required_path}")

WORK_DIR = TUTORIAL_ROOT / "notebooks/T05-DX-Compiler"
MODEL_DIR = WORK_DIR / "models"
CONFIG_DIR = WORK_DIR / "configs"
OUTPUT_DIR = WORK_DIR / "outputs"

for path in (WORK_DIR, MODEL_DIR, CONFIG_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

# Reuse the SDK sample images without copying them into this tutorial workspace.
CALIBRATION_SOURCE = DX_COM_DIR / "calibration_dataset"
CALIBRATION_DIR = WORK_DIR / "calibration_dataset"
if not CALIBRATION_SOURCE.is_dir():
    raise FileNotFoundError(f"Calibration dataset was not found: {CALIBRATION_SOURCE}")
if not CALIBRATION_DIR.exists():
    CALIBRATION_DIR.symlink_to(CALIBRATION_SOURCE, target_is_directory=True)

os.chdir(WORK_DIR)
print(f"DX-COM       : {DXCOM_PATH}")
print(f"Workspace    : {WORK_DIR}")
print(f"Calibration  : {CALIBRATION_DIR} -> {CALIBRATION_SOURCE}")


### 1.1 DX-COM installation options

`dx-all-suite/dx-compiler/install.sh` creates a dedicated compiler environment at `venv-dx-compiler-local`. This tutorial uses that environment explicitly, so it does not depend on the Jupyter kernel's Python environment.

DX-COM can also be installed easily from [PyPI](https://pypi.org/project/dx-com/) in a separate terminal:

```bash
uv venv ~/venv-dx-com
source ~/venv-dx-com/bin/activate
uv pip install --python ~/venv-dx-com/bin/python dx-com
dxcom --version
```

A PyPI installation provides DX-COM, but you still need the rest of the DEEPX SDK and a supported host to run the complete compile-and-deploy workflow.


The next cell is equivalent to these terminal commands:

```bash
source ~/dx-all-suite/dx-compiler/venv-dx-compiler-local/bin/activate
dxcom --version
dxcom -h
```

The actual SDK path may differ; the code cell uses the path loaded from `config.json`.


In [ ]:
!source "{DX_COMPILER_VENV}/bin/activate" && dxcom --version


In [ ]:
!source "{DX_COMPILER_VENV}/bin/activate" && dxcom -h


## 2. Export a small ONNX model

This project uses a `uv`-managed Jupyter environment, which may not contain the `pip` module. `uv pip install --python "{sys.executable}"` installs packages into the current Jupyter kernel explicitly. These packages are used to export and inspect ONNX. DX-COM itself continues to run from `DX_COMPILER_VENV`.


In [ ]:
import sys

!uv pip install --python "{sys.executable}" --quiet "torch>=2.0" torchvision onnx


In [ ]:
import torch
import torchvision
import onnx

torch.manual_seed(0)
weights = torchvision.models.MobileNet_V2_Weights.DEFAULT
model = torchvision.models.mobilenet_v2(weights=weights).eval()
dummy_input = torch.randn(1, 3, 224, 224)
mobilenet_onnx = MODEL_DIR / "mobilenet_v2.onnx"

torch.onnx.export(
    model,
    dummy_input,
    mobilenet_onnx,
    input_names=["input"],
    output_names=["output"],
    opset_version=18,
    do_constant_folding=True,
)
print(mobilenet_onnx)


### 2.1 Validate the ONNX contract

The input name in the JSON file must exactly match the ONNX graph input name. Shape inference and `onnx.checker` catch many export errors before compilation.


In [ ]:
onnx_model = onnx.load(mobilenet_onnx)
onnx.checker.check_model(onnx_model)
print("Opset:", [(item.domain or "ai.onnx", item.version) for item in onnx_model.opset_import])

def tensor_shape(value_info):
    return [dim.dim_value or dim.dim_param for dim in value_info.type.tensor_type.shape.dim]

print("Inputs:")
for value in onnx_model.graph.input:
    print(f"  {value.name}: {tensor_shape(value)}")
print("Outputs:")
for value in onnx_model.graph.output:
    print(f"  {value.name}: {tensor_shape(value)}")


You can also open the ONNX file in [Netron](https://netron.app/) to inspect tensor names, shapes, and operators. DX-TRON is deprecated in DX-COM 2.4.0; use Netron before compilation and the DX-COM HTML summary after compilation.


## 3. Create the calibration configuration

The configuration describes both the model input contract and how source images become input tensors.

| Field | Purpose |
|---|---|
| `inputs` | Exact ONNX input name and static shape |
| `calibration_method` | Observer used to estimate quantization ranges |
| `calibration_num` | Number of representative samples to use |
| `default_loader.dataset_path` | Directory containing calibration inputs |
| `preprocessings` | Ordered image-to-tensor transformations |

Preprocessing order matters. The values must match the preprocessing used when the original model was trained and evaluated.


In [ ]:
mobilenet_config = {
    "inputs": {"input": [1, 3, 224, 224]},
    "calibration_method": "ema",
    "calibration_num": 100,
    "default_loader": {
        "dataset_path": "./calibration_dataset",
        "file_extensions": ["jpeg", "jpg", "png", "JPEG"],
        "preprocessings": [
            {"resize": {"mode": "torchvision", "size": 256, "interpolation": "BILINEAR"}},
            {"centercrop": {"width": 224, "height": 224}},
            {"convertColor": {"form": "BGR2RGB"}},
            {"div": {"x": 255.0}},
            {"normalize": {
                "mean": [0.485, 0.456, 0.406],
                "std": [0.229, 0.224, 0.225]
            }},
            {"transpose": {"axis": [2, 0, 1]}},
            {"expandDim": {"axis": 0}}
        ]
    }
}

mobilenet_config_path = CONFIG_DIR / "mobilenet_v2.json"
mobilenet_config_path.write_text(json.dumps(mobilenet_config, indent=2) + "\n")
print(mobilenet_config_path.read_text())


## 4. Compile to DXNN

`--gen_log` preserves compiler logs and `--export_html` creates a model summary. The command keeps errors visible and uses a dedicated output directory.

The next cell is equivalent to these terminal commands:

```bash
source ~/dx-all-suite/dx-compiler/venv-dx-compiler-local/bin/activate
dxcom -m models/mobilenet_v2.onnx \
      -c configs/mobilenet_v2.json \
      -o outputs/mobilenet_v2_q_lite \
      --gen_log \
      --export_html
```


In [ ]:
mobilenet_output = OUTPUT_DIR / "mobilenet_v2_q_lite"

!source "{DX_COMPILER_VENV}/bin/activate" && \
  dxcom -m "{mobilenet_onnx}" \
        -c "{mobilenet_config_path}" \
        -o "{mobilenet_output}" \
        --gen_log \
        --export_html

In [ ]:
generated = sorted(path.relative_to(WORK_DIR) for path in mobilenet_output.rglob("*"))
print("\n".join(map(str, generated)))
dxnn_candidates = list(mobilenet_output.glob("*.dxnn"))
if not dxnn_candidates:
    raise FileNotFoundError("Compilation did not produce a DXNN file. Review the compiler output above.")
mobilenet_dxnn = dxnn_candidates[0]
print(f"\nDXNN: {mobilenet_dxnn}")


### 4.1 Inspect and benchmark

- `dxparse -v` reports the compiled model structure and tensor metadata.
- `dxrun --use-ort -t 5` runs a five-second synthetic-input benchmark. It measures runtime throughput; it does **not** validate task accuracy.
- A successful compile also does not prove that calibration or application preprocessing is correct. Validate accuracy with representative data before deployment.


In [ ]:
!dxparse -m "{mobilenet_dxnn}" -v


In [ ]:
!dxrun -m "{mobilenet_dxnn}" --use-ort -t 5


If the compiler reports `[INFO] Added nodes`, inspect which preprocessing operations were inserted into the graph. Do not apply the same normalization, color conversion, or transpose again in the runtime application.


## 5. Compile a model from the DEEPX Model Zoo

The [DEEPX Model Zoo](https://developer.deepx.ai/modelzoo/) provides searchable model metadata and downloadable artifacts, including ONNX models, DXNN models, and compiler JSON files for supported quantization variants.

This exercise uses **Resnet50** because its ONNX file is small. The workflow is the same for a larger model:

1. choose a model and quantization variant,
2. download its ONNX and matching JSON,
3. inspect the ONNX input contract,
4. adapt environment-specific JSON values, and
5. compile into a new output directory.


### 5.1. Download a Resnet50 ONNX file and a dxcom configuration file (json)

In [ ]:
MODELZOO_ONNX_URL = "https://sdk.deepx.ai/modelzoo/onnx/resnet50_224x224.onnx"
MODELZOO_JSON_URL = "https://sdk.deepx.ai/modelzoo/q-lite-json/2_4_0/resnet50_224x224.json"
MODELZOO_ONNX = MODEL_DIR / "resnet50_224x224.onnx"
MODELZOO_JSON = CONFIG_DIR / "resnet50_224x224.json"

!wget --continue --output-document="{MODELZOO_ONNX}" "{MODELZOO_ONNX_URL}"
!wget --continue --output-document="{MODELZOO_JSON}" "{MODELZOO_JSON_URL}"


In [ ]:
modelzoo_model = onnx.load(MODELZOO_ONNX)
onnx.checker.check_model(modelzoo_model)
print("ONNX inputs:")
for value in modelzoo_model.graph.input:
    print(f"  {value.name}: {tensor_shape(value)}")

downloaded_config = json.loads(MODELZOO_JSON.read_text())
print("Downloaded dataset path:", downloaded_config["default_loader"]["dataset_path"])
print("Configured input:", downloaded_config["inputs"])


### 5.2. Update the calibration path of dxcom configuration file (json)

Model Zoo JSON files contain the calibration setup used to produce the published model. Their dataset path belongs to the build environment and will usually not exist on your computer. Keep the model-specific preprocessing, but replace the dataset path with your local representative dataset.

The SDK sample images are used here only to make the compiler workflow reproducible. For an accuracy decision, use samples from the real deployment domain.


In [ ]:
local_modelzoo_config = downloaded_config.copy()
local_modelzoo_config["default_loader"] = downloaded_config["default_loader"].copy()
local_modelzoo_config["default_loader"]["dataset_path"] = "./calibration_dataset"

MODELZOO_LOCAL_JSON = CONFIG_DIR / "resnet50_224x224.json"
MODELZOO_LOCAL_JSON.write_text(json.dumps(local_modelzoo_config, indent=2) + "\n")
print(MODELZOO_LOCAL_JSON.read_text())


### 5.3. Compile

The next cell is equivalent to these terminal commands:

```bash
source ~/dx-all-suite/dx-compiler/venv-dx-compiler-local/bin/activate
dxcom -m models/resnet50_224x224.onnx \
      -c configs/resnet50_224x224.json \
      -o outputs/resnet50_224x224_q_lite \
      --gen_log \
      --export_html
```

In [ ]:
modelzoo_output = OUTPUT_DIR / "resnet50_224x224_q_lite"

!source "{DX_COMPILER_VENV}/bin/activate" && \
  dxcom -m "{MODELZOO_ONNX}" \
        -c "{MODELZOO_LOCAL_JSON}" \
        -o "{modelzoo_output}" \
        --gen_log \
        --export_html


### 5.4. Inspect and benchmark

In [ ]:
modelzoo_dxnn_files = list(modelzoo_output.glob("*.dxnn"))
if not modelzoo_dxnn_files:
    raise FileNotFoundError("No DXNN file was generated.")
modelzoo_dxnn = modelzoo_dxnn_files[0]
!dxparse -m "{modelzoo_dxnn}" -v
!dxrun -m "{modelzoo_dxnn}" --use-ort -t 5


### 5.5 Open the latest HTML compilation report

DX-COM creates an HTML model summary when `--export_html` is enabled. The next cell finds the most recently generated report under this tutorial's output directory and displays a button that opens it in a new browser tab.


In [ ]:
from html import escape
from urllib.parse import quote
from IPython.display import HTML, display

html_reports = [path for path in OUTPUT_DIR.rglob("*.html") if path.is_file()]
if not html_reports:
    raise FileNotFoundError(
        "No DX-COM HTML report was found. "
        "Compile a model with --export_html first."
    )

latest_report = max(html_reports, key=lambda path: path.stat().st_mtime)
relative_report = latest_report.resolve().relative_to(TUTORIAL_ROOT.resolve())
report_url = "/files/" + quote(relative_report.as_posix(), safe="/")

display(HTML(
    f'<a href="{escape(report_url, quote=True)}" target="_blank" '
    'rel="noopener noreferrer" '
    'style="display:inline-block;padding:10px 16px;background:#2563eb;'
    'color:white;text-decoration:none;border-radius:6px;font-weight:600;">'
    f'Open DX-COM report: {escape(latest_report.name)}'
    '</a>'
))
print(f"Report: {latest_report}")


## 6. Troubleshooting checklist

- **`dxcom: command not found`**: activate `DX_COMPILER_VENV`, or call `DXCOM_PATH` directly.
- **Input key error**: compare `inputs` in JSON with `model.graph.input` exactly.
- **Dynamic shape or batch error**: export a static shape with batch size 1.
- **No calibration files**: check the path, extensions, and file permissions.
- **Unexpected accuracy loss**: verify channel order, scaling, normalization, resize policy, and dataset representativeness.
- **Runtime preprocessing mismatch**: check whether DX-COM inserted preprocessing nodes.

Keep each experiment in its own output directory. This makes compiler reports and binaries traceable.


## Summary

You exported and validated ONNX, built a calibration configuration, compiled two DXNN models, and learned how to adapt a Model Zoo JSON file. Continue with the Intermediate notebook for calibration design, PPU configuration, Python API use, and YOLO26 TopK-based graph optimization.
